# xeus-lean + Sparkle HDL — interactive hardware tutorial

This is the **native** Lean Jupyter kernel with the [Sparkle HDL](https://github.com/Verilean/sparkle) library on the search path. Eleven short sections cover what xeus-lean + Sparkle can do today:

1. **Simulate** an 8-bit counter directly as a Lean function.
2. **Render an SVG waveform** inline in the notebook.
3. **Emit a VCD file** (open it in [GTKWave](http://gtkwave.sourceforge.net/) for a real waveform viewer).
4. **Synthesize SystemVerilog** from a Sparkle DSL function.
5. **Prove** a hardware property with `bv_decide` / `native_decide`.
6. **Visualize a real-ish protocol** — an I²C master writing one byte, with SCL and SDA on the same plot.
7. **Stateful circuit** — a 4-bit counter built with Sparkle's `circuit do ... let count ← Signal.reg ...` macro, evaluated through the C FFI memoized loop.
8. **Notebook helpers** — `#help_x`, `#findDecl`, `#listNs`, `#sig`, `#bash`.
9. **Mermaid sketches** — quick block-diagram drafts via `#mermaid`.
10. **Block-accurate diagrams** via `Display.blockDiagram` (trapezoid MUX, real cloud, AND/OR/NOT gates, ⊕ adder).
11. **Multi-million-tick traces** via `.wdb` — zstd-block compressed waveform DB, ~50× smaller than gzipped VCD, viewer streams only the visible window.

### Two REPL gotchas

The xlean kernel treats your whole session as one virtual `.lean` file. That has two consequences worth knowing up front:

1. **`import` only works in the very first cell you run.** Once any non-import command has executed, `import Sparkle` will be rejected with *"invalid 'import' command, it must be used in the beginning of the file"*. If you hit that, **Restart Kernel** and run the import cell first.
2. **`open` does NOT persist across cells.** Each cell is type-checked from a snapshot of the env that excludes prior `open`s. Repeat the `open Sparkle.Core.Domain Sparkle.Core.Signal` line at the top of every cell that uses short names.

In [ ]:
-- FIRST cell: imports go here, before any other command.
-- The first run loads Sparkle's full olean tree and takes ~90 s.
import Display
import Sparkle
import Sparkle.Compiler.Elab
import Sparkle.Backend.VCD
import Sparkle.Backend.Verilog

## 1. Simulate an 8-bit counter

A `Signal d α` in Sparkle is just a function from a clock tick (`Nat`) to a value (`α`). We can build one directly with the anonymous-constructor syntax `⟨fun t => ...⟩` and then evaluate it at any tick.

In [ ]:
open Sparkle.Core.Domain Sparkle.Core.Signal

-- An 8-bit counter: at clock tick t, the value is t mod 256.
def counter : Signal defaultDomain (BitVec 8) := ⟨fun t => BitVec.ofNat 8 t⟩

#eval (counter.val 0).toNat       -- 0
#eval (counter.val 5).toNat       -- 5
#eval (counter.val 255).toNat     -- 255
#eval (counter.val 256).toNat     -- 0  (BitVec wrap-around)

## 2. Render the counter as an SVG waveform

`Display.waveform` takes a list of `Nat` values and emits an SVG with `image/svg+xml` MIME, which JupyterLab renders inline. We sample the counter for 16 ticks and visualize them.

In [ ]:
open Sparkle.Core.Domain Sparkle.Core.Signal

def counterWave : Signal defaultDomain (BitVec 8) := ⟨fun t => BitVec.ofNat 8 t⟩
def samples : List Nat := (List.range 16).map fun t => (counterWave.val t).toNat

#eval Display.waveform "cnt[7:0]" samples 8 28 80

## 3. Emit a VCD file (for GTKWave)

The same trace can be written as a [Value Change Dump](https://en.wikipedia.org/wiki/Value_change_dump) — the format that GTKWave, Surfer, and most commercial waveform viewers consume. The cell below prints the VCD source; in a real flow you would either

- write the plain text:  `IO.FS.writeFile "counter.vcd" vcd`, or
- write a gzip-compressed `.vcd.gz` (~10× smaller, GTKWave reads it natively):  `Display.writeGz "counter.vcd.gz" vcd`

then `gtkwave counter.vcd[.gz]`.

In [ ]:
open Sparkle.Core.Domain Sparkle.Core.Signal Sparkle.Backend.VCD

def cntSig : Signal defaultDomain (BitVec 8) := ⟨fun t => BitVec.ofNat 8 t⟩

def writer  := (VCDWriter.new "counter").addVar "cnt" 8
def trace   := sampleBitVecSignal cntSig "cnt" 16
def vcd     := generateVCD writer trace

#eval IO.println vcd

## 4. Synthesize SystemVerilog from the Sparkle DSL

We write hardware as a Lean function over `Signal`s — a tiny ALU that adds and subtracts based on a select bit. Sparkle's elaborator walks the function and produces an IR `Module`, then `toVerilog` emits synthesizable SystemVerilog.

The cell below defines a `#showVerilog` macro that runs Sparkle's `synthesizeCombinational` pipeline and pipes the result through `Display.verilog`, which renders it with Highlight.js syntax coloring (loaded lazily from a CDN).

In [ ]:
-- One-time helper: a #showVerilog macro that runs Sparkle's
-- synthesizeCombinational pipeline and renders the resulting
-- SystemVerilog with syntax highlighting (Display.verilog wraps
-- Highlight.js with a github theme).
open Lean Lean.Elab.Command in
elab "#showVerilog " id:ident : command => do
  let declName ← liftCoreM <| Lean.resolveGlobalConstNoOverload id
  -- Synthesise the IR module inside TermElabM, then drop back to
  -- CoreM to fire Display.verilog (which is plain IO).
  let v ← liftTermElabM do
    let (module, _) ← Sparkle.Compiler.Elab.synthesizeCombinational declName
    pure (Sparkle.Backend.Verilog.toVerilog module)
  liftCoreM <| (Display.verilog v : IO Unit)

In [ ]:
open Sparkle.Core.Domain Sparkle.Core.Signal

-- Sparkle DSL: a tiny ALU. `sel = 0` → a + b, `sel = 1` → a - b.
def alu (sel : Signal Domain Bool)
        (a b : Signal Domain (BitVec 8)) : Signal Domain (BitVec 8) :=
  Signal.mux sel (a - b) (a + b)

#showVerilog alu

## 5. Prove a hardware property

The whole reason Sparkle lives inside Lean is *formal* hardware design — the same `BitVec` we simulate above also has a full proof theory. Lean's `bv_decide` tactic discharges any quantifier-free `BitVec` proposition by SAT; `native_decide` reduces decidable goals at native speed.

Below we prove four small properties — the wrap-around our 4-bit counter showed visually, commutativity of 8-bit addition, an XOR-with-0xFF identity, and a theorem about our `counter` simulation itself.

In [ ]:
-- Wrap-around: 255 + 1 = 0 in 8-bit arithmetic.
example : (255#8 + 1#8 = 0#8) := by native_decide

-- Commutativity of 8-bit addition (proved via bit-blasting → SAT).
example (a b : BitVec 8) : a + b = b + a := by bv_decide

-- A small ALU-style invariant: XOR with 0xFF flips every bit.
example (a : BitVec 8) : (a ^^^ 0xff#8) = ~~~ a := by bv_decide

-- And a theorem about the `counter` we simulated above:
-- `counter.val t` always equals `BitVec.ofNat 8 t`, by definition.
example (t : Nat) : counter.val t = BitVec.ofNat 8 t := rfl

## 6. A real-ish digital protocol: I²C master

So far our "waveform" was a monotonically-increasing counter — pretty, but uninteresting. Let's put both *clock* and *data* on the same plot and watch a small I²C master write the 7-bit address `0x21` (with R/W = 0, so the byte on the bus is `0x42`). Each bit period is `TPB = 8` simulation ticks, with SCL toggling at the half-period. The transaction is:

| bit period | what's happening |
|------------|------------------|
| 0          | START — SDA falls while SCL is high |
| 1..8       | data bits, MSB first (`0x42 = 0100_0010`) |
| 9          | ACK — slave pulls SDA low |
| 10         | STOP — SDA rises while SCL is high |

We build `scl` and `sda` as ordinary `Signal defaultDomain Bool` values, sample 11 bit periods (88 ticks), and render both lanes on one SVG with `Display.boolWave` — the two-lane digital-waveform helper that ships with xeus-lean.

In [ ]:
open Sparkle.Core.Domain Sparkle.Core.Signal

-- Bit-period geometry: 4 ticks SCL low, 4 ticks SCL high.
def TPH : Nat := 4
def TPB : Nat := 2 * TPH

-- Byte on the wire (7-bit address 0x21 + write bit). MSB first.
def addr : BitVec 8 := 0x42#8

-- Serial clock. High during START/STOP framing, toggles otherwise.
def scl : Signal defaultDomain Bool := ⟨fun t =>
  let bit := t / TPB
  let off := t % TPB
  if bit == 0 then true                          -- START framing
  else if bit ≥ 1 ∧ bit ≤ 9 then off ≥ TPH       -- data + ACK clocking
  else true                                       -- STOP framing
⟩

-- Serial data. START / data / ACK / STOP per the table above.
def sda : Signal defaultDomain Bool := ⟨fun t =>
  let bit := t / TPB
  let off := t % TPB
  if bit == 0 then off < TPH                     -- START: high then low
  else if bit ≥ 1 ∧ bit ≤ 8 then
    addr.getLsbD (7 - (bit - 1))                 -- MSB first
  else if bit == 9 then false                    -- slave ACK
  else if bit == 10 then off < TPH               -- STOP: low then high
  else false
⟩

def sclSamples : List Bool := (List.range 88).map (fun t => scl.val t)
def sdaSamples : List Bool := (List.range 88).map (fun t => sda.val t)

-- `Display.boolWave` ships with xeus-lean — render the two lanes on
-- one shared time axis.
#eval Display.boolWave [("SCL", sclSamples), ("SDA", sdaSamples)] 5 28

## 7. Bonus: a real stateful counter via `circuit do`

Sparkle's `circuit do ... let count ← Signal.reg 0#4 ; count <~ count + 1#4 ; return count` macro builds a stateful circuit through `Signal.loop`. Evaluating it from a notebook cell requires that the C FFI symbol `sparkle_eval_at` be reachable from the kernel; this image's `xlean` is linked against `libsparkle_barrier.a` (and the per-module Sparkle compiled objects) precisely so this works.

The cell below builds a 4-bit counter and samples it. `counter4.val 15 = 15` and `counter4.val 16 = 0` (BitVec wrap), and `#print axioms` confirms it depends on no axioms.

In [ ]:
open Sparkle.Core.Domain Sparkle.Core.Signal

def counter4 : Signal defaultDomain (BitVec 4) :=
  circuit do
    let count ← Signal.reg 0#4
    count <~ count + 1#4
    return count

#print axioms counter4
#eval (counter4.val 0).toNat        -- 0
#eval (counter4.val 1).toNat        -- 1
#eval (counter4.val 5).toNat        -- 5
#eval (counter4.val 15).toNat       -- 15
#eval (counter4.val 16).toNat       -- 0  (4-bit wrap)

## 8. Notebook helpers — find a function, run a shell command, see the menu

The kernel ships a few notebook-friendly helpers in `Display`:

* `#help_x` — list every `#`-command (or one specific command).
* `#findDecl "needle"` — substring-search the active env. AND-mode + paging.
* `#listNs SomeNamespace` — list declarations under a prefix.
* `#sig name` — show one declaration's type signature.
* `#bash "cmd"` — run a shell one-liner and dump output to the cell.

The cells below run each of these against the env we've already built.

In [ ]:
#help_x

In [ ]:
-- Substring search across the active env. Two-keyword AND-mode finds
-- only declarations whose name contains both "Signal" and "register".
#findDecl "Signal" "register" 0 6

In [ ]:
-- One declaration's type signature.
#sig Sparkle.Core.Signal.Signal.register

In [ ]:
-- Shell one-liner. Works through `bash -c` so pipes/redirects are fine.
#bash "uname -a; date -u +%FT%TZ; ls /app/.lake/build/bin"

## 9. Sketch a datapath with `#mermaid`

For quick block-diagram sketches `#mermaid` is enough — it pulls
`mermaid.js` from a CDN on first use and renders inline. Useful
shapes for hardware diagrams:

| Mermaid syntax | Meaning |
| -------------- | ------- |
| `[(R)]`        | register (cylinder) |
| `>ALU]`        | combinational logic (flag) |
| `[/in/]` `[\out\]` | I/O ports (parallelograms) |
| `((1#4))`      | constant (circle) |
| `-->`          | data wire (solid arrow) |
| `-.->`         | clock wire (dashed arrow) |
| `==>\|n\|`     | bus, with `n`-bit label |

The cell below shows the same pipelined datapath we'll build with the
structured emitter in section 10.

In [ ]:
#mermaid "
flowchart LR
  IN[/in/]
  R1[(R1)]
  ALU>ALU]
  R2[(R2)]
  OUT[/out/]
  CLK([CLK])
  IN  --> R1
  R1  --> ALU
  ALU ==>|8| R2
  R2  --> OUT
  CLK -.-> R1
  CLK -.-> R2
  classDef clk fill:#fce4ec,stroke:#c2185b
  class CLK clk
"

## 10. Block-accurate hardware diagram with `Display.blockDiagram`

When you want shapes Mermaid can't draw — trapezoid MUX, real cloud,
canonical AND/OR/NOT gate symbols, ⊕ adder — `Display.blockDiagram`
emits a hand-laid SVG. Nodes carry their kind (`reg`, `mux`, `cloud`,
`andG`, `orG`, `notG`, `adder`, `port`, `const`, `clk`, …) and an
explicit `(col, row)` so layout is predictable.

Edges are typed:

* `.data`     — solid arrow
* `.clock`    — pink dashed arrow (clock distribution)
* `.bus n`    — thick arrow, labelled with the `n`-bit width

In [ ]:
def myDP : Display.Diagram := {
  nodes := [
    { id := "in",  label := "in",  kind := .port,  col := 0, row := 0 },
    { id := "imm", label := "imm", kind := .const, col := 0, row := 1 },
    { id := "CLK", label := "CLK", kind := .clk,   col := 0, row := 2 },
    { id := "R1",  label := "R1",  kind := .reg,   col := 1, row := 0 },
    -- 2-input MUX: inputs := 2 spreads its left-edge pins, edges below
    -- target M.0 and M.1 explicitly so the lines don't merge.
    { id := "M",   label := "MUX", kind := .mux,   col := 2, row := 0, inputs := 2 },
    { id := "ALU", label := "ALU", kind := .cloud, col := 3, row := 0 },
    { id := "R2",  label := "R2",  kind := .reg,   col := 4, row := 0 },
    { id := "out", label := "out", kind := .port,  col := 5, row := 0 }
  ],
  edges := [
    { src := "in",  dst := "R1"  },
    { src := "R1",  dst := "M.0" },
    { src := "imm", dst := "M.1" },
    { src := "M",   dst := "ALU" },
    { src := "ALU", dst := "R2",  kind := .bus 8 },
    { src := "R2",  dst := "out" },
    -- Clock lines route through the rail at the bottom; they no longer
    -- cross any data wire.
    { src := "CLK", dst := "R1",  kind := .clock },
    { src := "CLK", dst := "R2",  kind := .clock }
  ]
}

#eval Display.blockDiagram myDP

## 11. Multi-million-tick VGA trace via `.wdb`

For traces too long to keep in memory (or send across the comm) we
write a `.wdb` — xeus-lean's own per-block, zstd-compressed waveform
format. `Display.writeWdb` walks the lanes once, packs each batch of
65 536 transitions into a zstd frame, and writes a tiny block index
on top. Reading is symmetric: `Display.waveformFromWdb` opens the
file and only decompresses the blocks that intersect the JS viewer's
current viewport.

Below we model 3 frames of VGA 640×480 timing — H_TOTAL = 800,
V_TOTAL = 525, so 3 × 800 × 525 = 1 260 000 clocks. Three lanes:
horizontal sync, vertical sync, and display-enable (`de`). On disk
the result is roughly 22 KB (≈ 50 × smaller than the equivalent
gzipped VCD). The viewer scrolls and zooms over the full 1.26 M
tick range without ever materialising the whole trace.

In [ ]:
-- Build the 3 VGA sync lanes as plain Nat → Bool functions.
-- H_TOTAL = 800, H_SYNC pulse = first 96 ticks
-- V_TOTAL = 525, V_SYNC pulse = first 2 lines
-- DE = active region 144 ≤ h < 784, 35 ≤ v < 515
def vgaLanes : List Display.WaveformSession.Lane := [
  { name := "hsync", sample := fun t => (t % 800) ≥ 96 },
  { name := "vsync", sample := fun t => ((t / 800) % 525) ≥ 2 },
  { name := "de",    sample := fun t =>
      let h := t % 800
      let v := (t / 800) % 525
      144 ≤ h ∧ h < 784 ∧ 35 ≤ v ∧ v < 515 }
]

-- 3 frames worth of clocks. Writing takes ~5 s (one Nat → Bool call
-- per tick per lane); the resulting file is ~22 KB.
#eval Display.writeWdb "/tmp/vga.wdb" vgaLanes 1260000

-- Open it. Same interactive canvas as section 6/7; here it serves a
-- 1.26 M-tick trace with no memory pressure.
#eval Display.waveformFromWdb "vga" "/tmp/vga.wdb"